# "Mark on face" images at different resolutions

Extract every image annotated with `mark on face` (case insensitive) from a COCO file and save it at several resolutions, to study how visible the defect is as the resolution drops.

## Imports

In [1]:
import cv2
import json
import math
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt

## Config

In [ ]:
DATA_DIR = Path(r"G:\B1375\DATA\Top")
ANNOTATIONS_FILE = DATA_DIR / "_annotations.coco.json"
OUTPUT_DIR = DATA_DIR / "mark_on_face_resolutions"
TARGET_CATEGORY = "mark on face"

# Real-world size of one pixel in the original photos
ORIGINAL_MM_PER_PX = 0.7

## Resolutions

Each resolution is defined by the real-world size (mm) of one pixel; the scale factor is `ORIGINAL_MM_PER_PX / mm_per_px`.

**Human eye at 1 m**: a normal eye (20/20) resolves ~1 arcminute, i.e. `1000 mm * tan(1/60 deg) ≈ 0.29 mm`. To be able to *see* a feature it needs ~2 samples, so the smallest resolvable detail is ~0.58 mm (one line pair). `eye_1m` below uses 0.29 mm/px as the eye's sampling pitch. Since the originals are 0.7 mm/px (coarser than the eye), this one is an upsample (no new information) and serves as reference; the rest of the resolutions simulate progressively coarser vision / cameras.

In [ ]:
EYE_MM_PER_PX = 1000 * math.tan(math.radians(1 / 60))  # 1 arcminute at 1 m
print(f"Eye resolution at 1 m: {EYE_MM_PER_PX:.3f} mm")

RESOLUTIONS_MM_PER_PX = {
    "eye_1m": EYE_MM_PER_PX,
    "0.7mm": 0.7,   # same as original, resaved
    "1.0mm": 1.0,
    "1.4mm": 1.4,
    "2.1mm": 2.1,
    "2.8mm": 2.8,
    "4.2mm": 4.2,
    "5.6mm": 5.6,
}

pd.DataFrame(
    [{"name": n, "mm_per_px": round(mm, 3), "scale": round(ORIGINAL_MM_PER_PX / mm, 3)} for n, mm in RESOLUTIONS_MM_PER_PX.items()]
)

## Load annotations and select images

In [ ]:
with open(ANNOTATIONS_FILE, encoding="utf-8") as f:
    coco = json.load(f)

categories = {c["id"]: c["name"] for c in coco["categories"]}
target_ids = {cid for cid, name in categories.items() if name.strip().lower() == TARGET_CATEGORY}
print("Categories:", categories)
print("Target category ids:", target_ids)

images = {im["id"]: im for im in coco["images"]}
anns_by_image = {}
for ann in coco["annotations"]:
    if ann["category_id"] in target_ids:
        anns_by_image.setdefault(ann["image_id"], []).append(ann)

print(f"{len(anns_by_image)} images with '{TARGET_CATEGORY}' out of {len(images)}")

## Helpers

Annotations use the segmentation polygon when available, otherwise the bbox (`x, y, w, h`).

In [ ]:
def ann_to_polygons(ann):
    """Return a list of Nx2 float arrays (original pixel coordinates)."""
    seg = ann.get("segmentation")
    if isinstance(seg, list) and len(seg) > 0 and all(isinstance(p, list) and len(p) >= 6 for p in seg):
        return [np.array(p, dtype=np.float32).reshape(-1, 2) for p in seg]
    x, y, w, h = ann["bbox"]
    return [np.array([[x, y], [x + w, y], [x + w, y + h], [x, y + h]], dtype=np.float32)]


def resize_image(img, scale):
    h, w = img.shape[:2]
    new_size = (max(1, round(w * scale)), max(1, round(h * scale)))
    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    return cv2.resize(img, new_size, interpolation=interp)


def draw_annotations(img, anns, scale=1.0, color=(0, 0, 255)):
    out = img.copy()
    for ann in anns:
        for poly in ann_to_polygons(ann):
            cv2.polylines(out, [np.round(poly * scale).astype(np.int32)], True, color, 2)
    return out

## Export

Per image, in `OUTPUT_DIR/<image_stem>/`:
- `original.<ext>`: untouched copy
- `<resolution>.png`: resized image (lossless)
- `<resolution>_annotated.png`: same, with the mark outline drawn
- `annotations.json`: annotations scaled to each resolution

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
records = []

for image_id, anns in tqdm(anns_by_image.items()):
    info = images[image_id]
    src = DATA_DIR / info["file_name"]
    img = cv2.imread(str(src))
    if img is None:
        print(f"Could not read {src}")
        continue

    out_dir = OUTPUT_DIR / src.stem
    out_dir.mkdir(exist_ok=True)
    cv2.imwrite(str(out_dir / f"original{src.suffix}"), img)

    scaled_anns = {}
    for name, mm in RESOLUTIONS_MM_PER_PX.items():
        scale = ORIGINAL_MM_PER_PX / mm
        resized = resize_image(img, scale)
        cv2.imwrite(str(out_dir / f"{name}.png"), resized)
        cv2.imwrite(str(out_dir / f"{name}_annotated.png"), draw_annotations(resized, anns, scale))

        # Mark size in pixels at this resolution (bbox of each annotation)
        sizes = []
        for ann in anns:
            _, _, w, h = ann["bbox"]
            sizes.append((w * scale, h * scale))
        scaled_anns[name] = [[[round(float(v), 2) for v in p.flatten()] for p in ann_to_polygons(ann)] for ann in anns]
        scaled_anns[name] = [[[round(v * scale, 2) for v in poly] for poly in a] for a in scaled_anns[name]]
        records.append({
            "image": src.name, "resolution": name, "mm_per_px": mm,
            "img_size": resized.shape[1::-1],
            "n_marks": len(anns),
            "min_mark_px": round(min(min(w, h) for w, h in sizes), 1),
            "min_mark_mm": round(min(min(w, h) for w, h in sizes) * mm, 1),
        })

    with open(out_dir / "annotations.json", "w") as f:
        json.dump(scaled_anns, f)

df = pd.DataFrame(records)
df.to_csv(OUTPUT_DIR / "summary.csv", index=False)
df.head(len(RESOLUTIONS_MM_PER_PX))

## Mark size per resolution

Smallest side of each mark's bbox, in pixels. Marks that fall below ~2 px at a given resolution are, in practice, not detectable.

In [ ]:
summary = df.groupby("resolution", sort=False).agg(
    mm_per_px=("mm_per_px", "first"),
    median_min_mark_px=("min_mark_px", "median"),
    pct_marks_below_2px=("min_mark_px", lambda s: round(100 * (s < 2).mean(), 1)),
)
summary

## Visual comparison

In [ ]:
def show_comparison(image_id, pad_mm=60):
    """Crop around the first mark and show it at every resolution (same physical size)."""
    info, anns = images[image_id], anns_by_image[image_id]
    img = cv2.imread(str(DATA_DIR / info["file_name"]))
    x, y, w, h = anns[0]["bbox"]
    pad = pad_mm / ORIGINAL_MM_PER_PX
    x0, y0 = int(max(0, x - pad)), int(max(0, y - pad))
    x1, y1 = int(min(img.shape[1], x + w + pad)), int(min(img.shape[0], y + h + pad))

    n = len(RESOLUTIONS_MM_PER_PX)
    fig, axes = plt.subplots(2, math.ceil(n / 2), figsize=(4 * math.ceil(n / 2), 8))
    for ax, (name, mm) in zip(axes.ravel(), RESOLUTIONS_MM_PER_PX.items()):
        scale = ORIGINAL_MM_PER_PX / mm
        crop = resize_image(img[y0:y1, x0:x1], scale)
        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB), interpolation="nearest")
        ax.set_title(f"{name} ({crop.shape[1]}x{crop.shape[0]})")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


for image_id in list(anns_by_image)[:3]:
    show_comparison(image_id)

## Notes / findings

_Fill in after looking at the comparison above: the coarsest resolution at which the marks are still visible, how it compares to the eye reference, and which resolution to use for training._